# 01 · Search modes — one cognee memory, queried many ways

`cognee.add` + `cognee.cognify` built a knowledge graph + embeddings from Wikipedia in AgensGraph (the `cognee_wiki` database). Here we ask the **same question through six `SearchType`s** to see what cognee's memory layer gives beyond plain RAG.

> Run `build.py` first.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # examples/demos
from _common import config
config.require_openai_key(); config.quiet()          # quiet cognee's verbose logs
config.configure("cognee_wiki")
import cognee
from cognee.modules.search.types import SearchType
from cognee.infrastructure.databases.graph import get_graph_engine

def _name(node):
    if isinstance(node, dict):
        return str(node.get("name") or (node.get("text") or "")[:40] or node.get("id") or "?")
    return str(node)[:40]

def render(results):
    if isinstance(results, (str, bytes)) or not isinstance(results, (list, tuple)):
        results = [results] if results else []
    for r in results[:4]:
        if isinstance(r, (tuple, list)) and len(r) == 3:
            src, edge, tgt = r
            rel = edge.get("relationship_name") if isinstance(edge, dict) else str(edge)
            print(f"   ({_name(src)}) -[{rel}]-> ({_name(tgt)})")
        elif isinstance(r, dict):
            print("   " + str(r.get("text") or r.get("name") or r)[:150])
        else:
            print("   " + str(r).strip().replace(chr(10), " ")[:550])
m = await (await get_graph_engine()).get_graph_metrics(include_optional=False)
print("knowledge graph:", m["num_nodes"], "nodes,", m["num_edges"], "edges")

knowledge graph: 5664 nodes, 12895 edges


## The six modes

`GRAPH_COMPLETION` (graph-aware) vs `RAG_COMPLETION` (chunks only, the baseline) vs `INSIGHTS` (triplets, no LLM) vs `CHUNKS` vs `SUMMARIES` vs `GRAPH_COMPLETION_COT` (chain-of-thought).

In [2]:
question = "What is anarchism, and what ideas, people, and movements is it connected to?"
for mode in ["GRAPH_COMPLETION", "RAG_COMPLETION", "GRAPH_COMPLETION_COT", "INSIGHTS", "CHUNKS", "SUMMARIES"]:
    print(f"\n### {mode}")
    render(await config.search(query_text=question, query_type=getattr(SearchType, mode)))


### GRAPH_COMPLETION


   Anarchism is a political philosophy and movement that opposes all forms of authority, advocating for the abolition of coercive institutions and the establishment of stateless societies based on voluntary associations. It is associated with libertarian socialism, which combines libertarian principles with socialist values, and is historically linked to movements such as the Paris Commune, and various workers' struggles in the 19th and early 20th centuries. Notable figures in anarchism include practitioners of different schools of thought and app

### RAG_COMPLETION


   Anarchism is a political philosophy that opposes all forms of unnecessary authority and seeks to abolish institutions that enforce coercion and hierarchy, such as nation-states and capitalism. It advocates for stateless societies and voluntary associations. Originating from the Enlightenment, modern anarchism gained prominence in the late 19th to early 20th centuries, influencing worker emancipation movements and participating in revolutions like the Paris Commune and the Spanish Civil War. The movement includes various schools of thought and s

### GRAPH_COMPLETION_COT


   Anarchism is a political philosophy and movement that questions all forms of authority and seeks to abolish coercive institutions, advocating for stateless societies and voluntary associations. It is typically associated with libertarian socialism and influenced by significant historical events and movements such as the Paris Commune, the Russian Civil War, and the Spanish Civil War. Anarchism has also been involved in various workers' struggles. Modern anarchism emerged during the Enlightenment and underwent significant development in the 19th

### INSIGHTS


   (# Anarchism

Anarchism is a political ph) -[contains]-> (anarchism)
   (anarchism) -[is_a]-> (political philosophy)
   (anarchism) -[is_a]-> (libertarian socialism)
   (anarchism) -[influenced]-> (workers struggles)

### CHUNKS


   # Anarchism

Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the instituti
   # Anarcho-capitalism

Anarcho-capitalism (colloquially: ancap or '"an-cap"') is an anti-statist, libertarian political philosophy and economic theory 
   # Arminianism

Arminianism is a movement of Protestantism initiated in the early 17th century, based on the theological ideas of the Dutch Reformed th
   # Agrarianism

Agrarianism is a social and political philosophy that has promoted subsistence agriculture, family farming, widespread property ownersh

### SUMMARIES


   Anarchism is a political ideology that questions all forms of authority and seeks to eliminate coercive institutions, advocating for stateless societi
   Anarcho-capitalism advocates for a stateless society where private property, free markets, and self-ownership are central. It promotes voluntary excha
   Angst is a feeling of fear or anxiety, closely related to feelings of apprehension and insecurity. The term originated from Danish and Norwegian words
   Agrarianism is a philosophy that champions subsistence farming, family agriculture, and property ownership, emphasizing political decentralization. Su


## How it was built

```python
await cognee.add(wiki_articles, dataset_name="wiki")
await cognee.cognify(["wiki"])   # LLM extracts entities + relationships, summarizes, embeds
```

Graph + vectors both live in one AgensGraph database. Re-run `build.py` to (re)build.